# 05d — Feature Engineering V4: Weather Deltas + Stress Indices

Final feature engineering iteration. Adds 3-hour weather change features, real-time stress indices, tail activity features, and airport fatigue index. Also includes experimental model training to validate feature signal.

**Input:** `dataset/merged_flights_fe.parquet`

**Output:** `dataset/merged_flights_fe_v2.parquet`

**Features added:** origin/dest_pressure_delta_3h, origin/dest_wind_speed_delta_3h, origin_dep_delay_rate_1h, dest_dep_delay_rate_1h, origin_stress_index, real_time_turn_gap, tail_delays_today, tail_active_hours, origin_pressure_drop_stress, airport_fatigue_index, day_of_year

In [ ]:
import pandas as pd

raw = pd.read_parquet('flights_raw.parquet', columns=None)
print(f"flights_raw.parquet: {raw.shape[0]:,} rows x {raw.shape[1]} cols")
print(f"\nColumns:\n{list(raw.columns)}")
print(f"\nFirst 3 rows FL_DATE range: {raw['FL_DATE'].min()} to {raw['FL_DATE'].max()}")
del raw

import os
weather_path = 'dataset/weather'
if os.path.exists(weather_path):
    for f in sorted(os.listdir(weather_path)):
        size = os.path.getsize(os.path.join(weather_path, f)) / (1024*1024)
        print(f"\n  {f:<40} {size:>8.1f} MB")

merged = pd.read_parquet('dataset/merged_flights_fe.parquet', columns=None)
print(f"\nmerged_flights.parquet: {merged.shape[0]:,} rows x {merged.shape[1]} cols")
print(f"Columns: {list(merged.columns)}")
del merged

flights_raw.parquet: 20,928,599 rows x 46 cols

Columns:
['YEAR', 'QUARTER', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'FL_DATE', 'OP_UNIQUE_CARRIER', 'OP_CARRIER_AIRLINE_ID', 'TAIL_NUM', 'OP_CARRIER_FL_NUM', 'ORIGIN', 'ORIGIN_CITY_NAME', 'ORIGIN_STATE_ABR', 'ORIGIN_STATE_NM', 'DEST', 'DEST_CITY_NAME', 'DEST_STATE_ABR', 'DEST_STATE_NM', 'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'DEP_DELAY_NEW', 'DEP_DEL15', 'DEP_TIME_BLK', 'TAXI_OUT', 'TAXI_IN', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY', 'ARR_DELAY_NEW', 'ARR_DEL15', 'ARR_TIME_BLK', 'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'CRS_ELAPSED_TIME', 'ACTUAL_ELAPSED_TIME', 'AIR_TIME', 'FLIGHTS', 'DISTANCE', 'DISTANCE_GROUP', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY']

First 3 rows FL_DATE range: 2023-01-01 00:00:00 to 2025-12-31 00:00:00

  isd-history.csv                               2.8 MB

  raw                                           0.0 MB

  weather_clean.parquet                        16.1

In [ ]:
weather = pd.read_parquet('dataset/weather/weather_clean.parquet')
print(f"Weather: {weather.shape[0]:,} rows x {weather.shape[1]} cols")
print(f"\nColumns: {list(weather.columns)}")
print(f"\nFirst 5 rows:")
print(weather.head())
print(f"\nDate range: {weather.iloc[:, 0].min()} to {weather.iloc[:, 0].max()}")

Weather: 2,322,436 rows x 14 cols

Columns: ['YEAR', 'MONTH', 'DAY', 'HOUR', 'TEMP', 'DEW_POINT', 'PRESSURE', 'WIND_DIR', 'WIND_SPEED', 'SKY_CONDITION', 'PRECIP_1HR', 'PRECIP_6HR', 'AIRPORT', 'DATETIME']

First 5 rows:
   YEAR  MONTH  DAY  HOUR  TEMP  DEW_POINT  PRESSURE  WIND_DIR  WIND_SPEED  \
0  2023      1    1     0  10.0        1.1    1013.7     250.0         4.1   
1  2023      1    1     1   8.9        1.1    1013.7     210.0         1.5   
2  2023      1    1     2   8.9        1.1    1014.1     230.0         2.6   
3  2023      1    1     3   6.7        1.1    1014.5     120.0         3.1   
4  2023      1    1     4   6.1        1.1    1014.6       0.0         0.0   

   SKY_CONDITION  PRECIP_1HR  PRECIP_6HR AIRPORT            DATETIME  
0            6.0         NaN         NaN     ABQ 2023-01-01 00:00:00  
1            NaN         0.0         NaN     ABQ 2023-01-01 01:00:00  
2            NaN         0.0         NaN     ABQ 2023-01-01 02:00:00  
3            8.0         0.0

## Weather Delta Features (3h Change)
Computes 3-hour pressure and wind speed changes per airport from NOAA ISD-Lite. Captures approaching storm fronts.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load weather data
weather = pd.read_parquet('dataset/weather/weather_clean.parquet')
print(f"Weather: {weather.shape[0]:,} rows x {weather.shape[1]} cols")
print(f"Airports: {weather['AIRPORT'].nunique()}")

# Ensure datetime is proper
weather['DATETIME'] = pd.to_datetime(weather['DATETIME'])
weather = weather.sort_values(['AIRPORT', 'DATETIME']).reset_index(drop=True)

print(f"Date range: {weather['DATETIME'].min()} to {weather['DATETIME'].max()}")
print("✓ Weather loaded")

Weather: 2,322,436 rows x 14 cols
Airports: 100
Date range: 2023-01-01 00:00:00 to 2025-08-27 09:00:00
✓ Weather loaded


In [ ]:
weather = weather.set_index('DATETIME')

deltas = []
for airport, grp in weather.groupby('AIRPORT'):
    grp = grp.sort_index()
    
    grp['pressure_3h_ago'] = grp['PRESSURE'].shift(3)
    grp['wind_speed_3h_ago'] = grp['WIND_SPEED'].shift(3)
    grp['temp_3h_ago'] = grp['TEMP'].shift(3)
    
    grp['pressure_delta_3h'] = grp['PRESSURE'] - grp['pressure_3h_ago']
    grp['wind_speed_delta_3h'] = grp['WIND_SPEED'] - grp['wind_speed_3h_ago']
    grp['temp_delta_3h'] = grp['TEMP'] - grp['temp_3h_ago']
    
    deltas.append(grp[['AIRPORT', 'pressure_delta_3h', 'wind_speed_delta_3h', 'temp_delta_3h']].reset_index())

weather_deltas = pd.concat(deltas, ignore_index=True)

print(f"Weather deltas computed: {weather_deltas.shape[0]:,} rows")
print(f"\nPressure delta 3h stats:")
print(weather_deltas['pressure_delta_3h'].describe())
print(f"\nWind speed delta 3h stats:")
print(weather_deltas['wind_speed_delta_3h'].describe())
print(f"\nNull rates:")
for col in ['pressure_delta_3h', 'wind_speed_delta_3h', 'temp_delta_3h']:
    null_pct = weather_deltas[col].isna().mean() * 100
    print(f"  {col}: {null_pct:.1f}%")

Weather deltas computed: 2,322,436 rows

Pressure delta 3h stats:
count    2.259181e+06
mean     4.886284e-04
std      1.490468e+01
min     -9.999000e+02
25%     -9.000000e-01
50%      1.000000e-01
75%      9.000000e-01
max      9.999000e+02
Name: pressure_delta_3h, dtype: float64

Wind speed delta 3h stats:
count    2.308406e+06
mean     4.509605e-04
std      2.870705e+00
min     -9.990000e+01
25%     -1.100000e+00
50%      0.000000e+00
75%      1.100000e+00
max      9.990000e+01
Name: wind_speed_delta_3h, dtype: float64

Null rates:
  pressure_delta_3h: 2.7%
  wind_speed_delta_3h: 0.6%
  temp_delta_3h: 0.6%


In [ ]:
weather = pd.read_parquet('dataset/weather/weather_clean.parquet')
weather['DATETIME'] = pd.to_datetime(weather['DATETIME'])
weather = weather.sort_values(['AIRPORT', 'DATETIME']).reset_index(drop=True)

weather.loc[weather['PRESSURE'].isin([0, 9999, 999.9, 99.9]) | (weather['PRESSURE'] < 900), 'PRESSURE'] = np.nan
weather.loc[weather['WIND_SPEED'].isin([99.9, 999.9]), 'WIND_SPEED'] = np.nan
weather.loc[weather['TEMP'].isin([999.9, 99.9]), 'TEMP'] = np.nan

print(f"After sentinel cleaning:")
print(f"  PRESSURE nulls: {weather['PRESSURE'].isna().mean()*100:.1f}%  range: {weather['PRESSURE'].min():.1f} to {weather['PRESSURE'].max():.1f}")
print(f"  WIND_SPEED nulls: {weather['WIND_SPEED'].isna().mean()*100:.1f}%  range: {weather['WIND_SPEED'].min():.1f} to {weather['WIND_SPEED'].max():.1f}")
print(f"  TEMP nulls: {weather['TEMP'].isna().mean()*100:.1f}%  range: {weather['TEMP'].min():.1f} to {weather['TEMP'].max():.1f}")

weather = weather.set_index('DATETIME')
deltas = []

for airport, grp in weather.groupby('AIRPORT'):
    grp = grp.sort_index()
    grp['pressure_delta_3h'] = grp['PRESSURE'] - grp['PRESSURE'].shift(3)
    grp['wind_speed_delta_3h'] = grp['WIND_SPEED'] - grp['WIND_SPEED'].shift(3)
    grp['temp_delta_3h'] = grp['TEMP'] - grp['TEMP'].shift(3)
    deltas.append(grp[['AIRPORT', 'pressure_delta_3h', 'wind_speed_delta_3h', 'temp_delta_3h']].reset_index())

weather_deltas = pd.concat(deltas, ignore_index=True)

print(f"\nCleaned pressure delta 3h:")
print(f"  range: {weather_deltas['pressure_delta_3h'].min():.1f} to {weather_deltas['pressure_delta_3h'].max():.1f}")
print(f"  mean: {weather_deltas['pressure_delta_3h'].mean():.3f}")
print(f"  std: {weather_deltas['pressure_delta_3h'].std():.3f}")

print(f"\nCleaned wind speed delta 3h:")
print(f"  range: {weather_deltas['wind_speed_delta_3h'].min():.1f} to {weather_deltas['wind_speed_delta_3h'].max():.1f}")
print(f"  mean: {weather_deltas['wind_speed_delta_3h'].mean():.3f}")
print(f"  std: {weather_deltas['wind_speed_delta_3h'].std():.3f}")

print(f"\nNull rates:")
for col in ['pressure_delta_3h', 'wind_speed_delta_3h', 'temp_delta_3h']:
    null_pct = weather_deltas[col].isna().mean() * 100
    print(f"  {col}: {null_pct:.1f}%")

After sentinel cleaning:
  PRESSURE nulls: 1.9%  range: 960.3 to 1049.3
  WIND_SPEED nulls: 0.3%  range: 0.0 to 34.0
  TEMP nulls: 0.4%  range: -41.7 to 60.6

Cleaned pressure delta 3h:
  range: -26.8 to 23.0
  mean: 0.000
  std: 1.432

Cleaned wind speed delta 3h:
  range: -22.6 to 29.9
  mean: 0.000
  std: 2.070

Null rates:
  pressure_delta_3h: 3.1%
  wind_speed_delta_3h: 0.6%
  temp_delta_3h: 0.7%


In [ ]:
flights = pd.read_parquet('dataset/merged_flights_fe.parquet', 
                          columns=['FL_DATE', 'DEP_HOUR', 'ORIGIN', 'DEST', 'ARR_DEL15'])

flights['FL_DATE'] = pd.to_datetime(flights['FL_DATE'])
flights['origin_weather_dt'] = flights['FL_DATE'] + pd.to_timedelta(flights['DEP_HOUR'], unit='h')
flights['dest_weather_dt'] = flights['FL_DATE'] + pd.to_timedelta(flights['DEP_HOUR'], unit='h')

weather_deltas['DATETIME'] = pd.to_datetime(weather_deltas['DATETIME'])

origin_deltas = weather_deltas.rename(columns={
    'AIRPORT': 'ORIGIN',
    'DATETIME': 'origin_weather_dt',
    'pressure_delta_3h': 'origin_pressure_delta_3h',
    'wind_speed_delta_3h': 'origin_wind_speed_delta_3h',
    'temp_delta_3h': 'origin_temp_delta_3h',
})

flights = flights.merge(origin_deltas, on=['ORIGIN', 'origin_weather_dt'], how='left')

dest_deltas = weather_deltas.rename(columns={
    'AIRPORT': 'DEST',
    'DATETIME': 'dest_weather_dt',
    'pressure_delta_3h': 'dest_pressure_delta_3h',
    'wind_speed_delta_3h': 'dest_wind_speed_delta_3h',
    'temp_delta_3h': 'dest_temp_delta_3h',
})

flights = flights.merge(dest_deltas, on=['DEST', 'dest_weather_dt'], how='left')

print(f"Flights: {flights.shape[0]:,}")
print(f"\nMatch rates:")
for col in ['origin_pressure_delta_3h', 'dest_pressure_delta_3h', 
            'origin_wind_speed_delta_3h', 'dest_wind_speed_delta_3h']:
    matched = flights[col].notna().mean() * 100
    print(f"  {col}: {matched:.1f}%")

print(f"\nDelay rate by origin_pressure_delta_3h quintile:")
flights['pressure_q'] = pd.qcut(flights['origin_pressure_delta_3h'], 5, labels=False, duplicates='drop')
signal = flights.groupby('pressure_q')['ARR_DEL15'].agg(['mean', 'count'])
signal.columns = ['delay_rate', 'count']
for q, row in signal.iterrows():
    bar = "█" * int(row['delay_rate'] * 100)
    print(f"  Q{int(q)}: {row['delay_rate']:.3f} ({row['count']:>10,})  {bar}")

print(f"\nDelay rate by origin_wind_speed_delta_3h quintile:")
flights['wind_q'] = pd.qcut(flights['origin_wind_speed_delta_3h'], 5, labels=False, duplicates='drop')
signal_w = flights.groupby('wind_q')['ARR_DEL15'].agg(['mean', 'count'])
signal_w.columns = ['delay_rate', 'count']
for q, row in signal_w.iterrows():
    bar = "█" * int(row['delay_rate'] * 100)
    print(f"  Q{int(q)}: {row['delay_rate']:.3f} ({row['count']:>10,})  {bar}")

del flights

Flights: 18,227,796

Match rates:
  origin_pressure_delta_3h: 89.1%
  dest_pressure_delta_3h: 89.2%
  origin_wind_speed_delta_3h: 91.2%
  dest_wind_speed_delta_3h: 91.2%

Delay rate by origin_pressure_delta_3h quintile:
  Q0: 0.256 (3,259,839.0)  █████████████████████████
  Q1: 0.217 (3,247,338.0)  █████████████████████
  Q2: 0.198 (3,348,912.0)  ███████████████████
  Q3: 0.194 (3,300,300.0)  ███████████████████
  Q4: 0.201 (3,087,528.0)  ████████████████████

Delay rate by origin_wind_speed_delta_3h quintile:
  Q0: 0.200 (3,489,791.0)  ███████████████████
  Q1: 0.201 (5,728,559.0)  ████████████████████
  Q2: 0.220 (1,383,667.0)  █████████████████████
  Q3: 0.222 (2,748,314.0)  ██████████████████████
  Q4: 0.245 (3,272,333.0)  ████████████████████████


## Merge Weather Deltas to Flights
Joins weather deltas using departure hour (origin) and arrival hour (destination).

In [ ]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
import gc

flights = pd.read_parquet('dataset/merged_flights_fe.parquet')
flights['FL_DATE'] = pd.to_datetime(flights['FL_DATE'])
print(f"Loaded: {flights.shape[0]:,} rows x {flights.shape[1]} cols")

flights['origin_weather_dt'] = flights['FL_DATE'] + pd.to_timedelta(flights['DEP_HOUR'], unit='h')
flights['dest_weather_dt'] = flights['FL_DATE'] + pd.to_timedelta(flights['ARR_HOUR'], unit='h')

flights = flights.merge(origin_deltas, on=['ORIGIN', 'origin_weather_dt'], how='left')
flights = flights.merge(dest_deltas, on=['DEST', 'dest_weather_dt'], how='left')
print(f"✓ Weather deltas merged (dest uses ARR_HOUR)")

flights['day_of_year'] = flights['FL_DATE'].dt.dayofyear

from datetime import date
holiday_ranges = []
for year in [2023, 2024, 2025]:
    holiday_ranges.extend([
        (date(year, 11, 20), date(year, 11, 28)),
        (date(year, 12, 20), date(year, 12, 31)),
        (date(year, 1, 1), date(year, 1, 3)),
        (date(year, 6, 30), date(year, 7, 7)),
        (date(year, 5, 23), date(year, 5, 30)),
        (date(year, 8, 30), date(year, 9, 5)),
        (date(year, 3, 10), date(year, 3, 22)),
    ])

flights['is_holiday_week'] = 0
fl_date = flights['FL_DATE'].dt.date
for start, end in holiday_ranges:
    flights.loc[(fl_date >= start) & (fl_date <= end), 'is_holiday_week'] = 1

print(f"✓ Holiday week: {flights['is_holiday_week'].mean()*100:.1f}% of flights")

hourly_origin = flights.groupby(['ORIGIN', 'FL_DATE', 'DEP_HOUR']).agg(
    dep_delayed=('ARR_DEL15', 'sum'),
    dep_total=('ARR_DEL15', 'count'),
).reset_index()

hourly_origin = hourly_origin.sort_values(['ORIGIN', 'FL_DATE', 'DEP_HOUR'])
hourly_origin['prev_hr_delayed'] = hourly_origin.groupby('ORIGIN')['dep_delayed'].shift(1)
hourly_origin['prev_hr_total'] = hourly_origin.groupby('ORIGIN')['dep_total'].shift(1)
hourly_origin['origin_dep_delay_rate_1h'] = hourly_origin['prev_hr_delayed'] / hourly_origin['prev_hr_total']

flights = flights.merge(
    hourly_origin[['ORIGIN', 'FL_DATE', 'DEP_HOUR', 'origin_dep_delay_rate_1h']],
    on=['ORIGIN', 'FL_DATE', 'DEP_HOUR'], how='left'
)
print(f"✓ Origin dep delay rate 1h: {flights['origin_dep_delay_rate_1h'].notna().mean()*100:.1f}% matched")

hourly_dest = flights.groupby(['DEST', 'FL_DATE', 'ARR_HOUR']).agg(
    arr_delayed=('ARR_DEL15', 'sum'),
    arr_total=('ARR_DEL15', 'count'),
).reset_index()

hourly_dest = hourly_dest.sort_values(['DEST', 'FL_DATE', 'ARR_HOUR'])
hourly_dest['prev_hr_delayed'] = hourly_dest.groupby('DEST')['arr_delayed'].shift(1)
hourly_dest['prev_hr_total'] = hourly_dest.groupby('DEST')['arr_total'].shift(1)
hourly_dest['dest_arr_delay_rate_1h'] = hourly_dest['prev_hr_delayed'] / hourly_dest['prev_hr_total']

flights = flights.merge(
    hourly_dest[['DEST', 'FL_DATE', 'ARR_HOUR', 'dest_arr_delay_rate_1h']],
    on=['DEST', 'FL_DATE', 'ARR_HOUR'], how='left'
)
print(f"✓ Dest arr delay rate 1h: {flights['dest_arr_delay_rate_1h'].notna().mean()*100:.1f}% matched")

new_features = [
    'origin_pressure_delta_3h', 'dest_pressure_delta_3h',
    'origin_wind_speed_delta_3h', 'dest_wind_speed_delta_3h',
    'day_of_year', 'is_holiday_week',
    'origin_dep_delay_rate_1h', 'dest_arr_delay_rate_1h',
]

print(f"\n{'=' * 60}")
print("SIGNAL CHECK — Delay rate spread across quintiles")
print(f"{'=' * 60}")

for feat in new_features:
    valid = flights[feat].notna().sum()
    if valid < 1000:
        print(f"\n{feat}: SKIPPED ({valid} values)")
        continue
    try:
        q = pd.qcut(flights[feat], 5, labels=False, duplicates='drop')
        signal = flights.groupby(q)['ARR_DEL15'].mean()
        spread = signal.max() - signal.min()
        status = "STRONG" if spread > 0.05 else "MODERATE" if spread > 0.02 else "WEAK"
        print(f"\n{feat}: spread={spread:.3f} [{status}]")
        for qi, rate in signal.items():
            bar = "█" * int(rate * 100)
            print(f"  Q{int(qi)}: {rate:.3f} {bar}")
    except Exception as e:
        print(f"\n{feat}: ERROR — {e}")

gc.collect()
print(f"\n✓ All features engineered. Dataset: {flights.shape[0]:,} rows")

Loaded: 18,227,796 rows x 116 cols
✓ Weather deltas merged (dest uses ARR_HOUR)
✓ Holiday week: 15.7% of flights
✓ Origin dep delay rate 1h: 100.0% matched
✓ Dest arr delay rate 1h: 100.0% matched

SIGNAL CHECK — Delay rate spread across quintiles

origin_pressure_delta_3h: spread=0.062 [STRONG]
  Q0: 0.256 █████████████████████████
  Q1: 0.217 █████████████████████
  Q2: 0.198 ███████████████████
  Q3: 0.194 ███████████████████
  Q4: 0.201 ████████████████████

dest_pressure_delta_3h: spread=0.076 [STRONG]
  Q0: 0.261 ██████████████████████████
  Q1: 0.229 ██████████████████████
  Q2: 0.198 ███████████████████
  Q3: 0.185 ██████████████████
  Q4: 0.187 ██████████████████

origin_wind_speed_delta_3h: spread=0.046 [MODERATE]
  Q0: 0.200 ███████████████████
  Q1: 0.201 ████████████████████
  Q2: 0.220 █████████████████████
  Q3: 0.222 ██████████████████████
  Q4: 0.245 ████████████████████████

dest_wind_speed_delta_3h: spread=0.038 [MODERATE]
  Q0: 0.209 ████████████████████
  Q1: 0.200

## Hourly Departure Delay Rate (1h Lookback)
Previous hour departure delay rate at origin and destination using DEP_DEL15. Point-in-time safe via shift(1).

In [ ]:
flights.drop(columns=['origin_dep_delay_rate_1h', 'dest_arr_delay_rate_1h'], inplace=True)

hourly_origin2 = flights.groupby(['ORIGIN', 'FL_DATE', 'DEP_HOUR']).agg(
    dep_delayed=('DEP_DEL15', 'sum'),
    dep_total=('DEP_DEL15', 'count'),
).reset_index()

hourly_origin2 = hourly_origin2.sort_values(['ORIGIN', 'FL_DATE', 'DEP_HOUR'])
hourly_origin2['prev_hr_delayed'] = hourly_origin2.groupby('ORIGIN')['dep_delayed'].shift(1)
hourly_origin2['prev_hr_total'] = hourly_origin2.groupby('ORIGIN')['dep_total'].shift(1)
hourly_origin2['origin_dep_delay_rate_1h'] = hourly_origin2['prev_hr_delayed'] / hourly_origin2['prev_hr_total']

flights = flights.merge(
    hourly_origin2[['ORIGIN', 'FL_DATE', 'DEP_HOUR', 'origin_dep_delay_rate_1h']],
    on=['ORIGIN', 'FL_DATE', 'DEP_HOUR'], how='left'
)

hourly_dest2 = flights.groupby(['DEST', 'FL_DATE', 'DEP_HOUR']).agg(
    dep_delayed=('DEP_DEL15', 'sum'),
    dep_total=('DEP_DEL15', 'count'),
).reset_index()

hourly_dest2 = hourly_dest2.sort_values(['DEST', 'FL_DATE', 'DEP_HOUR'])
hourly_dest2['prev_hr_delayed'] = hourly_dest2.groupby('DEST')['dep_delayed'].shift(1)
hourly_dest2['prev_hr_total'] = hourly_dest2.groupby('DEST')['dep_total'].shift(1)
hourly_dest2['dest_dep_delay_rate_1h'] = hourly_dest2['prev_hr_delayed'] / hourly_dest2['prev_hr_total']

flights = flights.merge(
    hourly_dest2[['DEST', 'FL_DATE', 'DEP_HOUR', 'dest_dep_delay_rate_1h']],
    on=['DEST', 'FL_DATE', 'DEP_HOUR'], how='left'
)

print("FIXED SIGNAL CHECK (using DEP_DEL15):")
for feat in ['origin_dep_delay_rate_1h', 'dest_dep_delay_rate_1h']:
    q = pd.qcut(flights[feat], 5, labels=False, duplicates='drop')
    signal = flights.groupby(q)['ARR_DEL15'].mean()
    spread = signal.max() - signal.min()
    status = "STRONG" if spread > 0.05 else "MODERATE" if spread > 0.02 else "WEAK"
    print(f"\n{feat}: spread={spread:.3f} [{status}]")
    for qi, rate in signal.items():
        bar = "█" * int(rate * 100)
        print(f"  Q{int(qi)}: {rate:.3f} {bar}")

FIXED SIGNAL CHECK (using DEP_DEL15):

origin_dep_delay_rate_1h: spread=0.261 [STRONG]
  Q0: 0.133 █████████████
  Q1: 0.174 █████████████████
  Q2: 0.243 ████████████████████████
  Q3: 0.394 ███████████████████████████████████████

dest_dep_delay_rate_1h: spread=0.238 [STRONG]
  Q0: 0.140 █████████████
  Q1: 0.173 █████████████████
  Q2: 0.241 ████████████████████████
  Q3: 0.378 █████████████████████████████████████


## Composite Stress Features
`origin_stress_index`: hourly congestion × departure delay rate.
`real_time_turn_gap`: scheduled buffer minus previous tail delay — top SHAP predictor.

In [ ]:
flights['origin_stress_index'] = flights['origin_dep_delay_rate_1h'] * flights['hourly_flight_count']
flights['real_time_turn_gap'] = flights['scheduled_turnaround_buffer'] - flights['prev_tail_arr_delay']

print(f"real_time_turn_gap:")
print(f"  Nulls: {flights['real_time_turn_gap'].isna().mean()*100:.1f}%")
print(f"  % negative (guaranteed late): {(flights['real_time_turn_gap'] < 0).mean()*100:.1f}%")

for feat in ['origin_stress_index', 'real_time_turn_gap']:
    valid = flights[feat].dropna()
    if len(valid) < 1000:
        print(f"\n{feat}: SKIPPED")
        continue
    q = pd.qcut(valid, 5, labels=False, duplicates='drop')
    signal = flights.loc[q.index].groupby(q)['ARR_DEL15'].mean()
    spread = signal.max() - signal.min()
    status = "STRONG" if spread > 0.05 else "MODERATE" if spread > 0.02 else "WEAK"
    print(f"\n{feat}: spread={spread:.3f} [{status}]")
    for qi, rate in signal.items():
        bar = "█" * int(rate * 100)
        print(f"  Q{int(qi)}: {rate:.3f} {bar}")

real_time_turn_gap:
  Nulls: 32.5%
  % negative (guaranteed late): 3.9%

origin_stress_index: spread=0.198 [STRONG]
  Q0: 0.150 ██████████████
  Q1: 0.188 ██████████████████
  Q2: 0.230 ██████████████████████
  Q3: 0.347 ██████████████████████████████████

real_time_turn_gap: spread=0.529 [STRONG]
  Q0: 0.628 ██████████████████████████████████████████████████████████████
  Q1: 0.146 ██████████████
  Q2: 0.105 ██████████
  Q3: 0.099 █████████
  Q4: 0.114 ███████████


## Tail Activity Features
`tail_delays_today`: cumulative delays for this aircraft today.
`tail_active_hours`: hours since first flight of the day.

In [ ]:
flights = flights.sort_values(['TAIL_NUM', 'FL_DATE', 'DEP_HOUR'])
flights['tail_delays_today'] = flights.groupby(['TAIL_NUM', 'FL_DATE'])['ARR_DEL15'].cumsum().shift(1)

flights['tail_delays_today'] = flights['tail_delays_today'].fillna(0)

print(f"tail_delays_today:")
print(f"  Nulls: {flights['tail_delays_today'].isna().mean()*100:.1f}%")
print(f"  Distribution:")
print(flights['tail_delays_today'].value_counts().sort_index().head(8))

q = pd.qcut(flights['tail_delays_today'].rank(method='first'), 5, labels=False, duplicates='drop')
signal = flights.groupby(q)['ARR_DEL15'].mean()
spread = signal.max() - signal.min()
status = "STRONG" if spread > 0.05 else "MODERATE" if spread > 0.02 else "WEAK"
print(f"\ntail_delays_today: spread={spread:.3f} [{status}]")
for qi, rate in signal.items():
    bar = "█" * int(rate * 100)
    print(f"  Q{int(qi)}: {rate:.3f} {bar}")

tail_delays_today:
  Nulls: 0.0%
  Distribution:
tail_delays_today
0.0    12749939
1.0     3185075
2.0     1356488
3.0      588464
4.0      234136
5.0       82730
6.0       24378
7.0        5285
Name: count, dtype: int64

tail_delays_today: spread=0.281 [STRONG]
  Q0: 0.129 ████████████
  Q1: 0.136 █████████████
  Q2: 0.133 █████████████
  Q3: 0.254 █████████████████████████
  Q4: 0.410 █████████████████████████████████████████


In [ ]:
flights = flights.sort_values(['TAIL_NUM', 'FL_DATE', 'DEP_HOUR'])

first_dep = flights.groupby(['TAIL_NUM', 'FL_DATE'])['DEP_HOUR'].transform('first')
flights['tail_active_hours'] = flights['DEP_HOUR'] - first_dep

print(f"tail_active_hours:")
print(f"  Nulls: {flights['tail_active_hours'].isna().mean()*100:.1f}%")
print(flights['tail_active_hours'].describe())

q = pd.qcut(flights['tail_active_hours'].rank(method='first'), 5, labels=False, duplicates='drop')
signal = flights.groupby(q)['ARR_DEL15'].mean()
spread = signal.max() - signal.min()
status = "STRONG" if spread > 0.05 else "MODERATE" if spread > 0.02 else "WEAK"
print(f"\ntail_active_hours: spread={spread:.3f} [{status}]")
for qi, rate in signal.items():
    bar = "█" * int(rate * 100)
    print(f"  Q{int(qi)}: {rate:.3f} {bar}")

tail_active_hours:
  Nulls: 0.0%
count    1.822780e+07
mean     5.983802e+00
std      4.940909e+00
min      0.000000e+00
25%      1.000000e+00
50%      5.000000e+00
75%      1.000000e+01
max      2.300000e+01
Name: tail_active_hours, dtype: float64

tail_active_hours: spread=0.149 [STRONG]
  Q0: 0.144 ██████████████
  Q1: 0.164 ████████████████
  Q2: 0.203 ████████████████████
  Q3: 0.259 █████████████████████████
  Q4: 0.293 █████████████████████████████


## Unpredictability Analysis
Flights with cascade_score=0 that still delay — proves fundamental ceiling of ML prediction for aviation delays.

In [ ]:
proof = pd.read_parquet('dataset/merged_flights_fe.parquet', 
    columns=['ARR_DEL15', 'cascade_score', 'CARRIER_DELAY', 'WEATHER_DELAY', 
             'NAS_DELAY', 'LATE_AIRCRAFT_DELAY', 'SECURITY_DELAY', 'FL_DATE'])

test = proof[pd.to_datetime(proof['FL_DATE']) >= '2025-01-01']
delayed = test[test['ARR_DEL15'] == 1]

print(f"Total delayed flights in test set: {len(delayed):,}")
print(f"\n--- Delayed flights with cascade_score = 0 (no prior tail warning) ---")
no_warning = delayed[delayed['cascade_score'] == 0]
print(f"Count: {len(no_warning):,} ({len(no_warning)/len(delayed)*100:.1f}% of all delays)")

total_delay_mins = (no_warning['CARRIER_DELAY'].fillna(0) + 
                    no_warning['WEATHER_DELAY'].fillna(0) + 
                    no_warning['NAS_DELAY'].fillna(0) + 
                    no_warning['LATE_AIRCRAFT_DELAY'].fillna(0) +
                    no_warning['SECURITY_DELAY'].fillna(0))

carrier = no_warning['CARRIER_DELAY'].fillna(0).sum()
weather = no_warning['WEATHER_DELAY'].fillna(0).sum()
nas = no_warning['NAS_DELAY'].fillna(0).sum()
late_aircraft = no_warning['LATE_AIRCRAFT_DELAY'].fillna(0).sum()
security = no_warning['SECURITY_DELAY'].fillna(0).sum()
total = carrier + weather + nas + late_aircraft + security

print(f"\nBreakdown of surprise delay causes:")
print(f"  CARRIER_DELAY:        {carrier/total*100:.1f}% (maintenance at gate, crew issues)")
print(f"  NAS_DELAY:            {nas/total*100:.1f}% (ATC decisions, ground stops)")
print(f"  LATE_AIRCRAFT_DELAY:  {late_aircraft/total*100:.1f}% (delays outside cascade window)")
print(f"  WEATHER_DELAY:        {weather/total*100:.1f}% (weather not captured in features)")
print(f"  SECURITY_DELAY:       {security/total*100:.1f}%")

print(f"\n{'=' * 60}")
print(f"CARRIER + NAS = {(carrier+nas)/total*100:.1f}% of surprise delays")
print(f"These are maintenance failures, crew problems, and ATC decisions")
print(f"that happen minutes before departure. No feature can predict them.")
print(f"{'=' * 60}")

del proof, test, delayed, no_warning

Total delayed flights in test set: 1,037,836

--- Delayed flights with cascade_score = 0 (no prior tail warning) ---
Count: 602,377 (58.0% of all delays)

Breakdown of surprise delay causes:
  CARRIER_DELAY:        42.9% (maintenance at gate, crew issues)
  NAS_DELAY:            28.3% (ATC decisions, ground stops)
  LATE_AIRCRAFT_DELAY:  20.8% (delays outside cascade window)
  WEATHER_DELAY:        7.9% (weather not captured in features)
  SECURITY_DELAY:       0.2%

CARRIER + NAS = 71.2% of surprise delays
These are maintenance failures, crew problems, and ATC decisions
that happen minutes before departure. No feature can predict them.


## Interaction Feature Candidates
Signal testing for pressure drop stress, turnaround stress, and seasonal interactions.

In [ ]:
flights['pressure_delta_both'] = flights['origin_pressure_delta_3h'] * flights['dest_pressure_delta_3h']

flights['turn_gap_negative'] = (flights['real_time_turn_gap'] < 0).astype(int)

flights['origin_pressure_drop_stress'] = flights['origin_pressure_delta_3h'].clip(upper=0).abs() * flights['hourly_flight_count']

flights['summer_pressure_drop'] = np.where(
    flights['day_of_year'].between(150, 250), 
    flights['origin_pressure_delta_3h'].clip(upper=0).abs(),
    0
)

flights['turn_gap_dest_stress'] = flights['real_time_turn_gap'] * -1 * flights['dest_inbound_arr_delay_3h'].fillna(0)

candidates = [
    'pressure_delta_both', 'turn_gap_negative',
    'origin_pressure_drop_stress', 'summer_pressure_drop',
    'turn_gap_dest_stress',
]

print("SIGNAL CHECK — Interaction candidates")
print("=" * 60)
for feat in candidates:
    valid = flights[feat].dropna()
    if len(valid) < 1000:
        print(f"\n{feat}: SKIPPED")
        continue
    try:
        q = pd.qcut(valid.rank(method='first'), 5, labels=False, duplicates='drop')
        signal = flights.loc[q.index].groupby(q)['ARR_DEL15'].mean()
        spread = signal.max() - signal.min()
        status = "STRONG" if spread > 0.05 else "MODERATE" if spread > 0.02 else "WEAK"
        print(f"\n{feat}: spread={spread:.3f} [{status}]")
        for qi, rate in signal.items():
            bar = "█" * int(rate * 100)
            print(f"  Q{int(qi)}: {rate:.3f} {bar}")
    except Exception as e:
        print(f"\n{feat}: ERROR — {e}")

SIGNAL CHECK — Interaction candidates

pressure_delta_both: spread=0.043 [MODERATE]
  Q0: 0.226 ██████████████████████
  Q1: 0.196 ███████████████████
  Q2: 0.194 ███████████████████
  Q3: 0.213 █████████████████████
  Q4: 0.237 ███████████████████████

turn_gap_negative: spread=0.167 [STRONG]
  Q0: 0.174 █████████████████
  Q1: 0.178 █████████████████
  Q2: 0.191 ███████████████████
  Q3: 0.179 █████████████████
  Q4: 0.341 ██████████████████████████████████

origin_pressure_drop_stress: spread=0.062 [STRONG]
  Q0: 0.193 ███████████████████
  Q1: 0.201 ████████████████████
  Q2: 0.198 ███████████████████
  Q3: 0.221 ██████████████████████
  Q4: 0.255 █████████████████████████

summer_pressure_drop: spread=0.069 [STRONG]
  Q0: 0.194 ███████████████████
  Q1: 0.196 ███████████████████
  Q2: 0.212 █████████████████████
  Q3: 0.195 ███████████████████
  Q4: 0.263 ██████████████████████████

turn_gap_dest_stress: spread=0.134 [STRONG]
  Q0: 0.251 █████████████████████████
  Q1: 0.227 █████

## Airport Fatigue Index
Cumulative departures delayed at origin airport today. Captures airport-level stress buildup.

In [ ]:
flights = flights.sort_values(['ORIGIN', 'FL_DATE', 'DEP_HOUR'])
flights['airport_fatigue_index'] = flights.groupby(['ORIGIN', 'FL_DATE'])['DEP_DEL15'].cumsum().shift(1)
flights['airport_fatigue_index'] = flights['airport_fatigue_index'].fillna(0)

print(f"airport_fatigue_index:")
print(flights['airport_fatigue_index'].describe())

q = pd.qcut(flights['airport_fatigue_index'].rank(method='first'), 5, labels=False, duplicates='drop')
signal = flights.groupby(q)['ARR_DEL15'].mean()
spread = signal.max() - signal.min()
status = "STRONG" if spread > 0.05 else "MODERATE" if spread > 0.02 else "WEAK"
print(f"\nairport_fatigue_index: spread={spread:.3f} [{status}]")
for qi, rate in signal.items():
    bar = "█" * int(rate * 100)
    print(f"  Q{int(qi)}: {rate:.3f} {bar}")

airport_fatigue_index:
count    1.822780e+07
mean     3.191561e+01
std      4.934288e+01
min      0.000000e+00
25%      3.000000e+00
50%      1.300000e+01
75%      4.100000e+01
max      7.330000e+02
Name: airport_fatigue_index, dtype: float64

airport_fatigue_index: spread=0.235 [STRONG]
  Q0: 0.120 ████████████
  Q1: 0.156 ███████████████
  Q2: 0.193 ███████████████████
  Q3: 0.238 ███████████████████████
  Q4: 0.356 ███████████████████████████████████


## Feature Definitions

In [ ]:
FEATURES_ORIG = [
    'MONTH', 'DAY_OF_WEEK', 'DEP_HOUR', 'ARR_HOUR', 'IS_HOLIDAY',
    'DISTANCE', 'profit_margin', 'origin_monthly_passengers',
    'origin_temp', 'origin_dew_point', 'origin_pressure',
    'origin_wind_dir', 'origin_wind_speed', 'origin_precip_1hr',
    'origin_weather_severity',
    'dest_temp', 'dest_dew_point', 'dest_pressure',
    'dest_wind_dir', 'dest_wind_speed', 'dest_precip_1hr',
    'dest_weather_severity',
    'airline_delay_rate_30d', 'origin_delay_rate_30d', 'dest_delay_rate_30d',
    'route_delay_rate_30d', 'origin_avg_taxi_out_30d',
    'flight_num_delay_rate_30d', 'origin_hour_delay_rate_30d',
    'carrier_origin_delay_rate_30d', 'dest_hour_delay_rate_30d',
    'airline_delay_rate_7d', 'origin_delay_rate_7d',
    'cascade_score', 'cascade_delay_minutes', 'hours_since_last_delay',
    'hourly_flight_count', 'scheduled_turnaround_buffer', 'tail_flight_num_today',
    'dest_hourly_flight_count',
    'inbound_arr_delay_3h', 'dest_inbound_arr_delay_3h',
    'prev_tail_arr_delay', 'national_hub_delay_2h',
    'OP_UNIQUE_CARRIER', 'ORIGIN', 'DEST', 'airline_cluster_label',
]

CAT_FEATURES = ['OP_UNIQUE_CARRIER', 'ORIGIN', 'DEST', 'airline_cluster_label']

## Experimental Model Training
Validates AUC improvement from 0.8578 (48 features) to 0.8629 (61 features).

In [ ]:
import time

NEW_FEATURES = [
    'origin_pressure_delta_3h', 'dest_pressure_delta_3h',
    'origin_wind_speed_delta_3h', 'dest_wind_speed_delta_3h',
    'day_of_year',
    'origin_dep_delay_rate_1h', 'dest_dep_delay_rate_1h',
    'origin_stress_index', 'real_time_turn_gap',
    'tail_delays_today', 'tail_active_hours',
    'origin_pressure_drop_stress', 'airport_fatigue_index',
]

ALL_FEATURES = FEATURES_ORIG + NEW_FEATURES

for col in CAT_FEATURES:
    flights[col] = flights[col].astype('category')

train = flights[flights['FL_DATE'] < '2025-01-01']
test  = flights[flights['FL_DATE'] >= '2025-01-01']

X_train = train[ALL_FEATURES]
y_train = train['ARR_DEL15']
X_test  = test[ALL_FEATURES]
y_test  = test['ARR_DEL15']

print(f"Features: {len(ALL_FEATURES)} ({len(FEATURES_ORIG)} original + {len(NEW_FEATURES)} new)")
print(f"Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}")

del train, test
gc.collect()

model_final = lgb.LGBMClassifier(
    objective='binary', metric='auc', is_unbalance=True,
    verbose=-1, random_state=42, n_jobs=-1,
    n_estimators=10000,
    learning_rate=0.009811877112556893,
    num_leaves=312,
    max_depth=13,
    min_child_samples=278,
    subsample=0.722267800117643,
    colsample_bytree=0.5308228457271158,
    reg_alpha=0.16398261731969382,
    reg_lambda=3.406904427387338,
    min_split_gain=0.9742015244622154,
    max_bin=175,
)

start = time.time()
model_final.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)],
)
elapsed = time.time() - start

y_pred_final = model_final.predict_proba(X_test)[:, 1]
auc_final = roc_auc_score(y_test, y_pred_final)

print(f"\n{'=' * 60}")
print(f"FINAL MODEL — {len(ALL_FEATURES)} Features")
print(f"  AUC:          {auc_final:.4f}")
print(f"  59-feature:   0.8623")
print(f"  Optuna 48:    0.8578")
print(f"  Baseline 48:  0.8572")
print(f"  Iteration:    {model_final.best_iteration_}")
print(f"  Time:         {elapsed/60:.1f} min")
print(f"{'=' * 60}")

imp = pd.DataFrame({
    'feature': ALL_FEATURES,
    'importance': model_final.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nFull feature rankings:")
for rank, (_, row) in enumerate(imp.iterrows(), 1):
    is_new = " NEW" if row['feature'] in NEW_FEATURES else ""
    print(f"  #{rank:<3} {row['feature']:<40} {row['importance']:>10,}{is_new}")

print(f"\n{'Threshold':<10} {'Precision':<12} {'Recall':<12} {'F1':<12} {'Flagged%':<10}")
print("-" * 55)
for t in [0.45, 0.50, 0.55, 0.60, 0.65, 0.70]:
    y_bin = (y_pred_final >= t).astype(int)
    tp = ((y_bin == 1) & (y_test == 1)).sum()
    fp = ((y_bin == 1) & (y_test == 0)).sum()
    fn = ((y_bin == 0) & (y_test == 1)).sum()
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    flagged = y_bin.mean() * 100
    print(f"  {t:.2f}     {prec:.4f}       {rec:.4f}       {f1:.4f}       {flagged:.1f}%")

Features: 61 (48 original + 13 new)
Train: 13,708,670  |  Test: 4,519,126
Training until validation scores don't improve for 100 rounds
[200]	valid_0's auc: 0.850256
[400]	valid_0's auc: 0.854625
[600]	valid_0's auc: 0.857028
[800]	valid_0's auc: 0.85848
[1000]	valid_0's auc: 0.859476
[1200]	valid_0's auc: 0.860161
[1400]	valid_0's auc: 0.86074
[1600]	valid_0's auc: 0.861181
[1800]	valid_0's auc: 0.861493
[2000]	valid_0's auc: 0.861752
[2200]	valid_0's auc: 0.86196
[2400]	valid_0's auc: 0.862126
[2600]	valid_0's auc: 0.862266
[2800]	valid_0's auc: 0.862378
[3000]	valid_0's auc: 0.862468
[3200]	valid_0's auc: 0.862543
[3400]	valid_0's auc: 0.862601
[3600]	valid_0's auc: 0.862642
[3800]	valid_0's auc: 0.862672
[4000]	valid_0's auc: 0.862705
[4200]	valid_0's auc: 0.862731
[4400]	valid_0's auc: 0.862757
[4600]	valid_0's auc: 0.862779
[4800]	valid_0's auc: 0.862801
[5000]	valid_0's auc: 0.862813
[5200]	valid_0's auc: 0.862832
[5400]	valid_0's auc: 0.862843
[5600]	valid_0's auc: 0.862851
[58

In [ ]:
import json
import joblib

joblib.dump(model_final, 'models/lgbm_delay_classifier_final.pkl')

with open('models/feature_list_final.txt', 'w') as f:
    for feat in ALL_FEATURES:
        f.write(feat + '\n')

metadata = {
    'auc': float(auc_final),
    'features': len(ALL_FEATURES),
    'best_iteration': int(model_final.best_iteration_),
    'train_rows': int(X_train.shape[0]),
    'test_rows': int(X_test.shape[0]),
    'categorical_features': CAT_FEATURES,
    'new_features': NEW_FEATURES,
    'params': {
        'learning_rate': 0.009811877112556893,
        'num_leaves': 312,
        'max_depth': 13,
        'min_child_samples': 278,
        'subsample': 0.722267800117643,
        'colsample_bytree': 0.5308228457271158,
        'reg_alpha': 0.16398261731969382,
        'reg_lambda': 3.406904427387338,
        'min_split_gain': 0.9742015244622154,
        'max_bin': 175,
    }
}

with open('models/model_final_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

size_mb = os.path.getsize('models/lgbm_delay_classifier_final.pkl') / 1024 / 1024
print(f"✓ Model saved: models/lgbm_delay_classifier_final.pkl ({size_mb:.1f} MB)")
print(f"✓ Features saved: models/feature_list_final.txt ({len(ALL_FEATURES)} features)")
print(f"✓ Metadata saved: models/model_final_metadata.json")
print(f"\nFinal AUC: {auc_final:.4f} | Features: {len(ALL_FEATURES)} | Iterations: {model_final.best_iteration_}")

✓ Model saved: models/lgbm_delay_classifier_final.pkl (275.3 MB)
✓ Features saved: models/feature_list_final.txt (61 features)
✓ Metadata saved: models/model_final_metadata.json

Final AUC: 0.8629 | Features: 61 | Iterations: 7162


In [21]:
flights.to_parquet('dataset/merged_flights_fe_v2.parquet')
print(f"✓ Saved: {flights.shape[0]:,} rows x {flights.shape[1]} cols")

✓ Saved: 18,227,796 rows x 138 cols
